<div style="
  background: #0d1f0f;
  border: 1px solid #1e4d22;
  border-radius: 12px;
  padding: 52px 48px 44px;
  font-family: 'Georgia', serif;
  position: relative;
  overflow: hidden;
">
  <div style="position:absolute;top:0;right:0;width:320px;height:320px;
    background:radial-gradient(circle at 80% 20%, #1a4d1e44 0%, transparent 65%);
    pointer-events:none;"></div>

  <div style="display:flex;align-items:flex-start;gap:20px;margin-bottom:28px;">
    <div style="font-size:52px;line-height:1;">🥾</div>
    <div>
      <div style="color:#4caf6c;font-family:'Courier New',monospace;font-size:11px;
        letter-spacing:4px;text-transform:uppercase;margin-bottom:8px;">
        Walk &amp; Hike Finder
      </div>
      <h1 style="color:#e8f0e9;font-size:38px;margin:0;font-weight:700;
        line-height:1.15;letter-spacing:-0.5px;">
        Hiking Agent<br>
        <span style="color:#4caf6c;">Notebook</span>
      </h1>
    </div>
  </div>

  <p style="color:#8db897;font-size:15.5px;line-height:1.8;max-width:660px;
    margin:0 0 32px;border-left:3px solid #2d6b35;padding-left:16px;">
    An end-to-end agentic pipeline that detects your location anywhere in the world,
    checks today's weather, discovers every kind of nearby walkable green area via
    OpenStreetMap, and asks a local LLM to recommend the best outing for the day.
    <strong style="color:#c8e6c9;">Fully tested, cell by cell.</strong>
  </p>

  <div style="display:flex;flex-wrap:wrap;gap:10px;">
    <span style="background:#1a3d1e;color:#81c784;padding:5px 13px;border-radius:20px;
      font-family:'Courier New',monospace;font-size:11.5px;border:1px solid #2e6b33;">
      📍 IP Geolocation</span>
    <span style="background:#1a3d1e;color:#81c784;padding:5px 13px;border-radius:20px;
      font-family:'Courier New',monospace;font-size:11.5px;border:1px solid #2e6b33;">
      🌤 Open-Meteo</span>
    <span style="background:#1a3d1e;color:#81c784;padding:5px 13px;border-radius:20px;
      font-family:'Courier New',monospace;font-size:11.5px;border:1px solid #2e6b33;">
      🗺 Overpass / OSM</span>
    <span style="background:#1a3d1e;color:#81c784;padding:5px 13px;border-radius:20px;
      font-family:'Courier New',monospace;font-size:11.5px;border:1px solid #2e6b33;">
      🤖 Ollama LLM</span>
    <span style="background:#1a3d1e;color:#81c784;padding:5px 13px;border-radius:20px;
      font-family:'Courier New',monospace;font-size:11.5px;border:1px solid #2e6b33;">
      🌍 Works Worldwide</span>
    <span style="background:#1a3d1e;color:#81c784;padding:5px 13px;border-radius:20px;
      font-family:'Courier New',monospace;font-size:11.5px;border:1px solid #2e6b33;">
      🆓 Zero API Cost</span>
  </div>
</div>

---
## 🏗 Architecture

```
┌──────────────────────────────────────────────────────────────────────────────┐
│                        HIKING AGENT  ·  PIPELINE OVERVIEW                    │
├──────────┬──────────┬───────────┬──────────┬──────────┬──────────┬──────────┤
│  STEP 1  │  STEP 2  │  STEP 3   │  STEP 4  │  STEP 5  │  STEP 6  │  STEP 7  │
│          │          │           │          │          │          │          │
│  Detect  │  Fetch   │  LLM ①    │  Query   │  Fetch   │  LLM ②   │  Chat    │
│ Location │ Weather  │  Go/No-Go │  Areas   │  Trails  │  Picks   │  Loop    │
│  via IP  │ Open-    │  yes/no   │ Overpass │  batch   │  + why   │  Q&A     │
│          │  Meteo   │           │   OSM    │  query   │          │          │
└──────────┴──────────┴───────────┴──────────┴──────────┴──────────┴──────────┘
                              ↓ if "no"
                        Agent exits early
```

| Module | Role | API | Key needed? |
|---|---|---|---|
| `location_eu.py` | IP -> lat/lon/city | geocoder | ✗ |
| `weather.py` | Hourly forecast | Open-Meteo | ✗ |
| `parks_eu.py` | Areas + trails | Overpass/OSM | ✗ |
| `main_eu.py` | Orchestration + LLM | Ollama (local) | ✗ |


---
## 📦 Step 0 - Install Dependencies

All libraries are open-source and free. Run once.


In [1]:
#%pip install geocoder requests ollama -q
import geocoder
import requests
import ollama
print("✅ Dependencies ready")

✅ Dependencies ready


---
## ⚙️ Step 1 - Configuration

Centralised settings. Change `MODEL` to whichever Ollama model you have pulled,
and tune `MAX_PARKS` or `SEARCH_RADIUS_KM` to control how many areas are surfaced.


In [2]:
import math, time, statistics, requests
from datetime import datetime, date

# ── LLM ──────────────────────────────────────────────────────────────────────
MODEL = "gpt-oss:120b-cloud"   # any model available in `ollama list`

# ── Overpass mirrors (tried in order on failure) ──────────────────────────────
OVERPASS_ENDPOINTS = [
    "https://overpass.kumi.systems/api/interpreter",
    "https://maps.mail.ru/osm/tools/overpass/api/interpreter",
    "https://overpass-api.de/api/interpreter",
]

# ── Search parameters ─────────────────────────────────────────────────────────
MAX_PARKS            = 6      # how many areas to show
SEARCH_RADIUS_KM     = 25     # park-search radius from your location
TRAIL_RADIUS_KM      = 10     # trail-search radius around each area centre
FALLBACK_RADIUS_KM   = 300    # max distance for fallback parks

print(f"Model           : {MODEL}")
print(f"Max areas       : {MAX_PARKS}")
print(f"Search radius   : {SEARCH_RADIUS_KM} km")
print(f"Trail radius    : {TRAIL_RADIUS_KM} km")

Model           : gpt-oss:120b-cloud
Max areas       : 6
Search radius   : 25 km
Trail radius    : 10 km


---
## 📍 Step 2 - Detect Location

Uses the `geocoder` library to resolve your public IP address to a city-level position (lat, lon, city, country). Accurate to ~10-50 km - more than enough for a 25 km park-search radius.

> **Privacy note:** Only your public IP is used. No GPS, no browser permissions.


In [3]:
def get_current_location():
    """Return (latitude, longitude, city, country) from IP geolocation."""
    try:
        g = geocoder.ip('me')
        if g.ok:
            return g.latlng[0], g.latlng[1], g.city, g.country
        print("Could not determine location from IP.")
        return None, None, None, None
    except Exception as e:
        print(f"Location error: {e}")
        return None, None, None, None

# ── Run ───────────────────────────────────────────────────────────────────────
latitude, longitude, city, country = get_current_location()

if latitude:
    print(f"📍 Location detected")
    print(f"   City    : {city}")
    print(f"   Country : {country}")
    print(f"   Coords  : {latitude:.4f}°N  {longitude:.4f}°E")
else:
    raise RuntimeError("Could not detect location - check network connection.")

📍 Location detected
   City    : Prague
   Country : CZ
   Coords  : 50.0880°N  14.4208°E


### 🧪 Location Test

Verify the returned coordinates are plausible by checking they fall within
expected ranges for a European city.


In [4]:
# ── TEST: sanity-check coordinates ───────────────────────────────────────────
assert latitude  is not None, "latitude must not be None"
assert longitude is not None, "longitude must not be None"
assert -90  <= latitude  <= 90,  f"latitude {latitude} out of range"
assert -180 <= longitude <= 180, f"longitude {longitude} out of range"
assert isinstance(city,    str) and len(city)    > 0, "city must be a non-empty string"
assert isinstance(country, str) and len(country) > 0, "country must be a non-empty string"

print("✅ All location assertions passed")
print(f"   lat={latitude}, lon={longitude}, city={city}, country={country}")

✅ All location assertions passed
   lat=50.088, lon=14.4208, city=Prague, country=CZ


---
## 🌤 Step 3 - Fetch Weather Forecast

[Open-Meteo](https://open-meteo.com) provides free, keyless, global hourly forecasts.
We request three variables - temperature, precipitation probability, and WMO weather code - then summarise today's **08:00-17:00 daylight window** into a single sentence for the LLM to evaluate.


In [22]:
WMO_CODES = {
    0: "Clear sky", 1: "Mainly clear", 2: "Partly cloudy", 3: "Overcast",
    45: "Fog", 48: "Depositing rime fog",
    51: "Light drizzle", 53: "Moderate drizzle", 55: "Dense drizzle",
    61: "Slight rain", 63: "Moderate rain", 65: "Heavy rain",
    71: "Slight snow fall", 73: "Moderate snow fall", 75: "Heavy snow fall",
    80: "Slight rain showers", 81: "Moderate rain showers", 82: "Violent rain showers",
    95: "Thunderstorm", 96: "Thunderstorm with slight hail", 99: "Thunderstorm with heavy hail",
}

def get_weather(lat, lon):
    """Fetch hourly forecast from Open-Meteo."""
    url = (
        f"https://api.open-meteo.com/v1/forecast"
        f"?latitude={lat}&longitude={lon}"
        f"&hourly=temperature_2m,precipitation_probability,weathercode"
    )
    r = requests.get(url, timeout=10)
    r.raise_for_status()
    return r.json()

def get_todays_weather_summary(data):
    """Summarise 08:00–17:00 today into one sentence."""
    h     = data.get("hourly", {})
    times = h.get("time", [])
    temps = h.get("temperature_2m", [])
    prec  = h.get("precipitation_probability", [])
    codes = h.get("weathercode", [])

    dt, dp, dc = [], [], []
    for i, ts in enumerate(times):
        d = datetime.fromisoformat(ts)
        if d.date() == date.today() and 8 <= d.hour <= 17:
            dt.append(temps[i]); dp.append(prec[i]); dc.append(codes[i])

    if not dt:
        return "Could not build a weather summary for today."

    desc = WMO_CODES.get(statistics.mode(dc), "variable weather")
    return (
        f"Today's forecast: {desc}, average {round(statistics.mean(dt))}°C, "
        f"max precipitation probability {max(dp)}%."
    )

# ── Run ───────────────────────────────────────────────────────────────────────
weather_data    = get_weather(latitude, longitude)
weather_summary = get_todays_weather_summary(weather_data)
print(f"🌤 {weather_summary}")

🌤 Today's forecast: Slight rain, average 15°C, max precipitation probability 95%.


### 🧪 Weather Tests

In [23]:
# ── TEST: weather data structure ─────────────────────────────────────────────
assert isinstance(weather_data, dict),            "weather_data must be a dict"
assert "hourly" in weather_data,                  "must contain 'hourly' key"
assert "temperature_2m" in weather_data["hourly"],"must contain temperature data"
assert "weathercode"    in weather_data["hourly"],"must contain weathercode data"

# ── TEST: summary is a non-empty string ──────────────────────────────────────
assert isinstance(weather_summary, str) and len(weather_summary) > 10
assert "°C"  in weather_summary, "summary should include temperature"
assert "%"   in weather_summary, "summary should include precipitation probability"

# ── TEST: 7-day hourly data has the expected volume ──────────────────────────
n_hours = len(weather_data["hourly"]["time"])
assert n_hours >= 24, f"expected at least 24 hourly records, got {n_hours}"

print(f"✅ Weather tests passed  ({n_hours} hourly records received)")

✅ Weather tests passed  (168 hourly records received)


---
## 🛜 Step 4 - Overpass API Helper

A thin wrapper that tries three public Overpass mirrors in sequence.
If one returns a 504 or times out, it moves silently to the next.
This makes every downstream query resilient to single-mirror outages.


In [7]:
def _post_overpass(query, timeout=45):
    """Try each Overpass mirror in turn; return parsed JSON or None."""
    for endpoint in OVERPASS_ENDPOINTS:
        host = endpoint.split('/')[2]
        try:
            r = requests.post(endpoint, data={"data": query}, timeout=timeout)
            if r.status_code == 504:
                print(f"  ⚠  504 from {host}, trying next mirror…")
                continue
            r.raise_for_status()
            return r.json()
        except requests.exceptions.Timeout:
            print(f"  ⚠  Timeout from {host}, trying next mirror…")
        except requests.exceptions.RequestException as e:
            print(f"  ⚠  Error from {host}: {e}")
    return None

print("✅ Overpass helper defined")

✅ Overpass helper defined


### 🧪 Overpass Connectivity Test

In [8]:
# ── TEST: basic Overpass connectivity ────────────────────────────────────────
# Minimal query - asks for one node near Prague; should always return quickly.
test_query = """
[out:json][timeout:10];
node["natural"="peak"](around:50000,50.0755,14.4378);
out 1;
"""
test_result = _post_overpass(test_query, timeout=15)

if test_result is not None:
    print(f"✅ Overpass reachable - got {len(test_result.get('elements', []))} element(s)")
else:
    print("⚠  All Overpass mirrors unreachable right now - fallback list will be used")

✅ Overpass reachable - got 0 element(s)


---
## 📐 Step 5 - Distance Calculation (Haversine)

All distance sorting - for both live Overpass results and the offline fallback list - uses the **haversine formula**, giving accurate great-circle distances anywhere on Earth.


In [9]:
def _haversine_km(lat1, lon1, lat2, lon2):
    """Great-circle distance in km between two (lat, lon) points."""
    R = 6371.0
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlam = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlam/2)**2
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

print("✅ Haversine function defined")

✅ Haversine function defined


### 🧪 Haversine Tests

In [10]:
# ── TEST: known distances ────────────────────────────────────────────────────
# Prague → Vienna  ≈ 252 km
prague_vienna = _haversine_km(50.0755, 14.4378, 48.2082, 16.3738)
assert 245 < prague_vienna < 260, f"Prague–Vienna: expected ~252 km, got {prague_vienna:.1f}"

# Prague → Berlin  ≈ 280 km
prague_berlin = _haversine_km(50.0755, 14.4378, 52.5200, 13.4050)
assert 270 < prague_berlin < 295, f"Prague–Berlin: expected ~280 km, got {prague_berlin:.1f}"

# Same point → 0
same = _haversine_km(50.0, 14.0, 50.0, 14.0)
assert same == 0.0, f"Same point should be 0 km, got {same}"

print(f"✅ Haversine tests passed")
print(f"   Prague → Vienna : {prague_vienna:.1f} km  (expected ≈ 252)")
print(f"   Prague → Berlin : {prague_berlin:.1f} km  (expected ≈ 280)")
print(f"   Same point      : {same:.1f} km")

✅ Haversine tests passed
   Prague → Vienna : 250.9 km  (expected ≈ 252)
   Prague → Berlin : 281.1 km  (expected ≈ 280)
   Same point      : 0.0 km


---
## 🌲 Step 6 - Discover Nearby Green Areas

Queries Overpass for **all types** of walkable green areas within `SEARCH_RADIUS_KM`:
national parks, nature reserves, protected areas, named forests, regional parks,
and local parks. Small, unknown local spots are included intentionally.

### Why broad tags?

The original approach filtered by `protect_class`, which excluded hundreds of
small Czech reserves. The 429 rate-limit problem from fetching trails per-park is
solved separately (Step 7 batches them). So we keep the query broad and just
tighten the search radius.

### Deduplication

OSM sometimes tags the same physical area twice - once as `leisure=nature_reserve`
and once as `boundary=protected_area` - with slightly different names
(e.g. *"Letenský profil"* and *"přírodní památka Letenský profil"*).
Entries within **100 m** of each other are collapsed into one, keeping the
higher-priority type label and the shorter name.


In [11]:
FALLBACK_PARKS = [
    # Czech & immediate neighbours
    {"name": "Bohemian Switzerland NP",   "lat": 50.8833, "lon": 14.2333},
    {"name": "Krkonoše NP",               "lat": 50.7333, "lon": 15.5833},
    {"name": "Šumava NP",                 "lat": 49.0333, "lon": 13.5000},
    {"name": "Podyjí NP",                 "lat": 48.8500, "lon": 15.8833},
    {"name": "Saxon Switzerland NP",      "lat": 50.9167, "lon": 14.1333},
    {"name": "Bavarian Forest NP",        "lat": 49.0333, "lon": 13.4333},
    {"name": "Tatra NP",                  "lat": 49.1667, "lon": 20.1167},
    # Western Europe
    {"name": "Berchtesgaden NP",          "lat": 47.6000, "lon": 12.9333},
    {"name": "Hohe Tauern NP",            "lat": 47.1000, "lon": 12.7500},
    {"name": "Swiss NP",                  "lat": 46.6667, "lon": 10.1833},
    {"name": "Gran Paradiso NP",          "lat": 45.5167, "lon":  7.2167},
    {"name": "Vanoise NP",                "lat": 45.3833, "lon":  6.9167},
    {"name": "Pyrenees NP",               "lat": 42.8333, "lon": -0.1167},
    {"name": "Picos de Europa NP",        "lat": 43.1833, "lon": -4.8333},
    {"name": "Peneda-Gerês NP",           "lat": 41.7833, "lon": -8.1500},
    # UK & Scandinavia
    {"name": "Cairngorms NP",             "lat": 57.1000, "lon": -3.5833},
    {"name": "Lake District NP",          "lat": 54.4500, "lon": -3.0833},
    {"name": "Jotunheimen NP",            "lat": 61.6000, "lon":  8.5000},
    # Eastern Europe & Balkans
    {"name": "Białowieża NP",             "lat": 52.7000, "lon": 23.8667},
    {"name": "Plitvice Lakes NP",         "lat": 44.8667, "lon": 15.6167},
    {"name": "Rila NP",                   "lat": 42.2000, "lon": 23.5833},
    {"name": "Triglav NP",                "lat": 46.3667, "lon": 13.8333},
    # North America
    {"name": "Yellowstone NP",            "lat": 44.4280, "lon":-110.5885},
    {"name": "Yosemite NP",               "lat": 37.8651, "lon":-119.5383},
    {"name": "Banff NP",                  "lat": 51.4968, "lon":-115.9281},
    {"name": "Great Smoky Mountains NP",  "lat": 35.6118, "lon": -83.4895},
    # Latin America
    {"name": "Torres del Paine NP",       "lat":-50.9423, "lon": -73.4068},
    {"name": "Iguazú NP",                 "lat":-25.6953, "lon": -54.4367},
    # Africa
    {"name": "Serengeti NP",              "lat": -2.3333, "lon":  34.8333},
    {"name": "Kruger NP",                 "lat":-23.9884, "lon":  31.5547},
    {"name": "Table Mountain NP",         "lat":-34.0000, "lon":  18.4167},
    # Asia
    {"name": "Fuji-Hakone-Izu NP",        "lat": 35.3606, "lon": 138.7274},
    {"name": "Zhangjiajie NP",            "lat": 29.3167, "lon": 110.4333},
    {"name": "Royal Chitwan NP",          "lat": 27.5000, "lon":  84.3333},
    {"name": "Khao Yai NP",               "lat": 14.4386, "lon": 101.3697},
    # Oceania
    {"name": "Blue Mountains NP",         "lat":-33.7167, "lon": 150.3167},
    {"name": "Fiordland NP",              "lat":-45.4167, "lon": 167.7167},
    {"name": "Tongariro NP",              "lat":-39.2000, "lon": 175.5667},
]

TYPE_PRIORITY = {
    "National Park":  5,
    "Nature Reserve": 4,
    "Regional Park":  3,
    "Forest":         2,
    "Park":           1,
    "Protected Area": 0,
}

def _parse_areas(data, user_lat, user_lon):
    """Parse Overpass response; dedup by name and by proximity (100 m)."""
    TYPE_MAP = {
        "national_park":  "National Park",
        "protected_area": "Protected Area",
        "regional_park":  "Regional Park",
    }
    areas, seen_names = [], set()
    for el in data.get("elements", []):
        tags = el.get("tags", {})
        name = tags.get("name") or tags.get("name:en")
        if not name or name in seen_names:
            continue
        seen_names.add(name)
        if "center" in el:
            lat, lon = el["center"].get("lat"), el["center"].get("lon")
        else:
            lat, lon = el.get("lat"), el.get("lon")
        if lat is None or lon is None:
            continue
        boundary = tags.get("boundary", "")
        leisure  = tags.get("leisure", "")
        landuse  = tags.get("landuse", "")
        if boundary in TYPE_MAP:
            atype = TYPE_MAP[boundary]
        elif leisure == "nature_reserve":
            atype = "Nature Reserve"
        elif leisure == "park":
            atype = "Park"
        elif landuse == "forest":
            atype = "Forest"
        else:
            atype = "Protected Area"
        areas.append({
            "name":        name,
            "type":        atype,
            "lat":         lat,
            "lon":         lon,
            "website":     tags.get("website") or tags.get("contact:website", ""),
            "distance_km": round(_haversine_km(user_lat, user_lon, lat, lon), 1),
        })

    # Proximity dedup: collapse entries within 100 m
    merged, skip = [], set()
    for i, a in enumerate(areas):
        if i in skip:
            continue
        for j, b in enumerate(areas):
            if j <= i or j in skip:
                continue
            if _haversine_km(a["lat"], a["lon"], b["lat"], b["lon"]) <= 0.1:
                if TYPE_PRIORITY.get(b["type"], 0) > TYPE_PRIORITY.get(a["type"], 0):
                    a["type"] = b["type"]
                if len(b["name"]) < len(a["name"]):
                    a["name"] = b["name"]
                skip.add(j)
        merged.append(a)
    return merged


def get_parks(lat, lon, radius_km=25):
    """
    Fetch all walkable green areas within radius_km, sorted nearest-first.
    Uses broad OSM tags - national parks, reserves, forests, local parks.
    Falls back to the global hardcoded list (distance-filtered) if Overpass fails.
    """
    radius_m = radius_km * 1000
    query = f"""
    [out:json][timeout:40];
    (
      node["boundary"="national_park"](around:{radius_m},{lat},{lon});
      way["boundary"="national_park"](around:{radius_m},{lat},{lon});
      relation["boundary"="national_park"](around:{radius_m},{lat},{lon});
      node["leisure"="nature_reserve"](around:{radius_m},{lat},{lon});
      way["leisure"="nature_reserve"](around:{radius_m},{lat},{lon});
      relation["leisure"="nature_reserve"](around:{radius_m},{lat},{lon});
      node["boundary"="protected_area"](around:{radius_m},{lat},{lon});
      way["boundary"="protected_area"](around:{radius_m},{lat},{lon});
      relation["boundary"="protected_area"](around:{radius_m},{lat},{lon});
      node["landuse"="forest"]["name"](around:{radius_m},{lat},{lon});
      way["landuse"="forest"]["name"](around:{radius_m},{lat},{lon});
      relation["landuse"="forest"]["name"](around:{radius_m},{lat},{lon});
      relation["boundary"="regional_park"](around:{radius_m},{lat},{lon});
      relation["leisure"="park"]["name"](around:{radius_m},{lat},{lon});
    );
    out center tags;
    """
    data  = _post_overpass(query, timeout=45)
    parks = _parse_areas(data, lat, lon) if data else []

    if len(parks) < 3:
        print(f"  Live query: {len(parks)} area(s) - filling from global fallback…")
        live_names = {p["name"] for p in parks}
        for fp in FALLBACK_PARKS:
            d = _haversine_km(lat, lon, fp["lat"], fp["lon"])
            if fp["name"] not in live_names and d <= FALLBACK_RADIUS_KM:
                parks.append({
                    "name": fp["name"], "type": "National Park",
                    "lat": fp["lat"],   "lon": fp["lon"],
                    "website": "",      "distance_km": round(d, 1),
                })

    parks.sort(key=lambda p: p["distance_km"])
    result = parks[:MAX_PARKS]

    if result:
        print(f"\n🌲 Found {len(result)} nearest area(s):")
        for p in result:
            print(f"   {p['distance_km']:>6.1f} km  [{p['type']}]  {p['name']}")
    else:
        print(f"No areas found within {radius_km} km.")
    return result

parks = get_parks(latitude, longitude, SEARCH_RADIUS_KM)

  ⚠  Timeout from overpass.kumi.systems, trying next mirror…
  ⚠  Error from maps.mail.ru: 403 Client Error: Forbidden for url: https://maps.mail.ru/osm/tools/overpass/api/interpreter
  ⚠  504 from overpass-api.de, trying next mirror…
  Live query: 0 area(s) - filling from global fallback…

🌲 Found 6 nearest area(s):
     89.4 km  [National Park]  Bohemian Switzerland NP
     94.4 km  [National Park]  Saxon Switzerland NP
    109.2 km  [National Park]  Krkonoše NP
    134.8 km  [National Park]  Šumava NP
    137.2 km  [National Park]  Bavarian Forest NP
    173.5 km  [National Park]  Podyjí NP


### 🧪 Parks Tests

In [12]:
# ── TEST: result structure ────────────────────────────────────────────────────
assert isinstance(parks, list),         "parks must be a list"
assert len(parks) > 0,                  "must find at least one area"
assert len(parks) <= MAX_PARKS,         f"must not exceed MAX_PARKS={MAX_PARKS}"

for i, p in enumerate(parks):
    assert "name"        in p, f"park[{i}] missing 'name'"
    assert "type"        in p, f"park[{i}] missing 'type'"
    assert "lat"         in p, f"park[{i}] missing 'lat'"
    assert "lon"         in p, f"park[{i}] missing 'lon'"
    assert "distance_km" in p, f"park[{i}] missing 'distance_km'"
    assert p["distance_km"] >= 0, f"park[{i}] distance must be non-negative"

# ── TEST: sorted nearest-first ────────────────────────────────────────────────
distances = [p["distance_km"] for p in parks]
assert distances == sorted(distances), f"Parks not sorted by distance: {distances}"

# ── TEST: proximity dedup worked (no two parks within 50 m of each other) ─────
for i in range(len(parks)):
    for j in range(i+1, len(parks)):
        d = _haversine_km(parks[i]["lat"], parks[i]["lon"],
                          parks[j]["lat"], parks[j]["lon"])
        assert d > 0.05, (
            f"Parks '{parks[i]['name']}' and '{parks[j]['name']}' "
            f"are only {d*1000:.0f} m apart - dedup may have missed them"
        )

print(f"✅ Parks tests passed ({len(parks)} areas, sorted, deduplicated)")
for p in parks:
    print(f"   {p['distance_km']:>6.1f} km  [{p['type']}]  {p['name']}")

✅ Parks tests passed (6 areas, sorted, deduplicated)
     89.4 km  [National Park]  Bohemian Switzerland NP
     94.4 km  [National Park]  Saxon Switzerland NP
    109.2 km  [National Park]  Krkonoše NP
    134.8 km  [National Park]  Šumava NP
    137.2 km  [National Park]  Bavarian Forest NP
    173.5 km  [National Park]  Podyjí NP


---
## 🥾 Step 7 - Fetch Hiking Trails (Single Batched Query)

Instead of one Overpass request per park (which caused cascading 429 rate-limit
errors when 500+ areas were returned), we build **one union query** that covers
all park centres simultaneously. Overpass handles the spatial join server-side.

Trails are returned as a flat list and then assigned to their nearest park
centre using a fast Euclidean approximation.

### SAC difficulty scale

| OSM value | Label |
|---|---|
| `hiking` | Easy |
| `mountain_hiking` | Moderate |
| `demanding_mountain_hiking` | Challenging |
| `alpine_hiking` | Hard |
| `demanding_alpine_hiking` / `difficult_alpine_hiking` | Very Hard / Expert |


In [13]:
SAC_LABELS = {
    "hiking":                    "Easy",
    "mountain_hiking":           "Moderate",
    "demanding_mountain_hiking": "Challenging",
    "alpine_hiking":             "Hard",
    "demanding_alpine_hiking":   "Very Hard",
    "difficult_alpine_hiking":   "Expert",
}

def get_trails_for_parks(parks, radius_km=10):
    """
    Fetch hiking trails for ALL parks in ONE Overpass query.
    Assign each trail to its nearest park centre.
    Returns dict: {park_name: [trail_dict, ...]}
    """
    if not parks:
        return {}

    radius_m = radius_km * 1000
    clauses = "\n".join(
        f'relation["route"="hiking"](around:{radius_m},{p["lat"]},{p["lon"]});'
        for p in parks
    )
    query = f"""
    [out:json][timeout:55];
    ( {clauses} );
    out center tags;
    """
    print("🔍 Fetching trails (single batched request)…")
    time.sleep(1)
    data = _post_overpass(query, timeout=60)

    if not data:
        print("  ⚠  Could not fetch trails - areas will be shown without trail detail.")
        return {p["name"]: [] for p in parks}

    def _qdist(lat1, lon1, lat2, lon2):
        if None in (lat1, lon1, lat2, lon2): return float("inf")
        dlat = lat1 - lat2
        dlon = (lon1 - lon2) * math.cos(math.radians((lat1+lat2)/2))
        return dlat**2 + dlon**2

    all_trails, seen = [], set()
    for el in data.get("elements", []):
        tags = el.get("tags", {})
        name = tags.get("name") or tags.get("name:en")
        if not name or name in seen: continue
        seen.add(name)
        center = el.get("center", {})
        try:
            tdist = round(float(tags.get("distance") or tags.get("length","")), 1)
        except (ValueError, TypeError):
            tdist = None
        all_trails.append({
            "name":        name,
            "lat":         center.get("lat"),
            "lon":         center.get("lon"),
            "distance_km": tdist,
            "difficulty":  SAC_LABELS.get(tags.get("sac_scale",""), "Unknown"),
            "surface":     tags.get("surface", "unknown"),
        })

    result = {p["name"]: [] for p in parks}
    for t in all_trails:
        nearest = min(parks, key=lambda p: _qdist(t["lat"], t["lon"], p["lat"], p["lon"]))
        result[nearest["name"]].append(t)

    total = sum(len(v) for v in result.values())
    print(f"✅ {total} trail(s) fetched and assigned across {len(parks)} area(s)")
    for p in parks:
        n = len(result[p["name"]])
        if n: print(f"   {p['name']}: {n} trail(s)")
    return result

trails_by_park = get_trails_for_parks(parks, TRAIL_RADIUS_KM)

🔍 Fetching trails (single batched request)…
  ⚠  504 from overpass.kumi.systems, trying next mirror…
  ⚠  Error from maps.mail.ru: 403 Client Error: Forbidden for url: https://maps.mail.ru/osm/tools/overpass/api/interpreter
  ⚠  504 from overpass-api.de, trying next mirror…
  ⚠  Could not fetch trails - areas will be shown without trail detail.


### 🧪 Trails Tests

In [14]:
# ── TEST: result is a dict keyed by park name ─────────────────────────────────
assert isinstance(trails_by_park, dict), "trails_by_park must be a dict"
assert set(trails_by_park.keys()) == {p["name"] for p in parks}, \
    "trails_by_park must have an entry for every park"

# ── TEST: each trail entry has required fields ────────────────────────────────
for park_name, trails in trails_by_park.items():
    assert isinstance(trails, list), f"trails for '{park_name}' must be a list"
    for t in trails:
        assert "name"        in t, f"trail missing 'name' in park '{park_name}'"
        assert "difficulty"  in t, f"trail missing 'difficulty'"
        assert "surface"     in t, f"trail missing 'surface'"
        if t["distance_km"] is not None:
            assert t["distance_km"] >= 0, "trail distance must be non-negative"

total_trails = sum(len(v) for v in trails_by_park.values())
print(f"✅ Trails tests passed - {total_trails} trail(s) across {len(parks)} area(s)")

✅ Trails tests passed - 0 trail(s) across 6 area(s)


---
## 🤖 Step 8 - LLM Decision ①: Weather Gate

The first of two LLM calls. We send the one-sentence weather summary and ask
for a binary **yes / no**. If the answer is *no*, the agent would stop here
and skip all the Overpass work - saving unnecessary API calls on rainy days.

In this notebook we continue regardless so you can see the full pipeline.


In [15]:
def query_model(system_prompt, user_prompt, messages=None):
    """Send prompt to Ollama; return (response_text, updated_history)."""
    if messages is None:
        messages = []
    messages.append({"role": "user", "content": user_prompt})
    if system_prompt and not any(d.get("role") == "system" for d in messages):
        messages.insert(0, {"role": "system", "content": system_prompt})
    response = ollama.chat(model=MODEL, messages=messages)
    content  = response["message"]["content"].strip()
    messages.append({"role": "assistant", "content": content})
    return content, messages

weather_decision, _ = query_model(
    "You are an assistant that determines if the weather is good for a walk or hike outside. "
    "Respond with only 'yes' or 'no'.",
    f"Is it a good day to go outside? {weather_summary}",
)

icon = "✅" if "yes" in weather_decision.lower() else "❌"
print(f"{icon} Model decision: {weather_decision}")
print()
if "no" in weather_decision.lower():
    print("⛔ In production the agent would stop here.")
    print("   Continuing anyway so the full notebook can be demonstrated.")
else:
    print("🥾 Weather approved - continuing to recommendations.")

❌ Model decision: no

⛔ In production the agent would stop here.
   Continuing anyway so the full notebook can be demonstrated.


---
## 🤖 Step 9 - LLM Decision ②: Recommendations

The main LLM call. We build a structured prompt with every area and its trails
(type, distance from user, trail difficulty, surface), then ask the model to act
as a friendly local guide and pick the best **2-3 options** with practical reasons.

The full conversation history is stored in `message_history` for the Q&A loop.


In [16]:
# ── Build structured prompt ───────────────────────────────────────────────────
prompt_data = ""
for park in parks:
    trails = trails_by_park.get(park["name"], [])
    prompt_data += f"\n[{park['type']}] {park['name']} - {park['distance_km']} km from you"
    if park.get("website"):
        prompt_data += f" ({park['website']})"
    prompt_data += "\n"
    if trails:
        for t in trails:
            dist = f"{t['distance_km']} km" if t["distance_km"] else "unknown length"
            prompt_data += (
                f"  • {t['name']} | {t['difficulty']} | {dist} | "
                f"surface: {t['surface']}\n"
            )
    else:
        prompt_data += "  • No marked trails in OSM - area is walkable.\n"

# ── Ask the model ─────────────────────────────────────────────────────────────
system = (
    f"You are a friendly, knowledgeable local guide for {city}, {country}. "
    "The user wants somewhere nearby to walk or hike today - nothing too far. "
    "From the list below, recommend the top 2–3 options. "
    "Favour closeness unless a slightly further spot is clearly better. "
    "For each pick: what kind of walk it is, roughly how long, and why it's worth it. "
    "Mention distance from the user where known. Be warm and practical, not encyclopaedic."
)

recommendations, message_history = query_model(
    system,
    f"Here are walkable green areas near {city} right now:\n{prompt_data}",
)

print("=" * 62)
print("🥾  RECOMMENDATIONS FOR TODAY")
print("=" * 62)
print(recommendations)

🥾  RECOMMENDATIONS FOR TODAY
Sure thing! All of the spots you listed are a bit of a drive, but they’re each worth a day out if you’re up for a little road‑time. Here are my top picks, ordered by closeness (and a touch of “wow‑factor”) so you can decide which one feels right for today.

---

### 1. **Bohemian Switzerland National Park**
- **Distance:** ≈ 89 km (about a 1‑hour‑15‑minute drive southeast of Prague)
- **What kind of walk?** Gentle to moderate forest trails with spectacular rock formations, gorge views and plenty of photo‑ops. The classic “Hřensko – Labský důl” loop is a great entry point.
- **Rough length:** 3–5 km (2–3 hours at a relaxed pace, plus time for a coffee by the river)
- **Why it’s worth it:** You’ll wander through fairy‑tale sandstone cliffs, the famous Pravčická brána (the biggest natural arch in Europe), and the dramatic narrow “Křižanka” gorge. The trails are well‑marked and the area feels close enough to feel like a quick escape, yet far enough to feel like

### 🧪 Recommendations Test

In [17]:
# ── TEST: recommendations is a non-trivial string ────────────────────────────
assert isinstance(recommendations, str),   "recommendations must be a string"
assert len(recommendations) > 80,          "recommendations seem too short"

# ── TEST: conversation history was populated correctly ────────────────────────
assert isinstance(message_history, list),  "message_history must be a list"
assert len(message_history) >= 3,          "history should have system + user + assistant"
roles = [m["role"] for m in message_history]
assert "user"      in roles, "history must contain a user message"
assert "assistant" in roles, "history must contain an assistant message"

print("✅ Recommendations tests passed")
print(f"   Response length   : {len(recommendations)} chars")
print(f"   History messages  : {len(message_history)}")

✅ Recommendations tests passed
   Response length   : 3299 chars
   History messages  : 3


---
## 💬 Step 10 - Follow-up Q&A Loop

The full message history is passed on every call so the model retains context
across turns. A second reflection call checks whether the model believes it has
fully answered the question - this is printed as a transparency cue, not a gate.

Run the cell and type questions. Enter `exit` to stop.

> **Example questions to try:**
> - *"How do I get to Kampa by metro?"*
> - *"Which one is best for dogs?"*
> - *"How long is the walk at Letenský profil?"*


In [18]:
def is_final_answer(messages):
    """
    Ask the model to reflect: was the last response a complete answer?
    Returns False silently on any error — it's a transparency cue, not a gate.
    """
    try:
        ref_msgs = messages[:] + [{
            "role": "system",
            "content": (
                "Has the last assistant response fully answered the user's question? "
                "Reply only 'yes' or 'no'."
            )
        }]
        resp = ollama.chat(model=MODEL, messages=ref_msgs)
        return "yes" in resp["message"]["content"].strip().lower()
    except Exception:
        return False   # fail silently — never interrupt the conversation


print("💬 Ask follow-up questions. Type 'exit' to stop.")
print("   Tip: type  remember: <fact>  to pin something for this session.\n")

memory = []   # pinned facts — survive history trimming

try:
    while True:
        follow_up = input("You > ").strip()

        if not follow_up:
            continue

        if follow_up.lower() in ("exit", "quit"):
            print("👋 Enjoy your walk!")
            break

        # ── remember: command ──────────────────────────────────────────────
        if follow_up.lower().startswith("remember:"):
            fact = follow_up[len("remember:"):].strip()
            if fact:
                memory.append(fact)
                print(f"\n📌 Pinned: \"{fact}\"\n")
            else:
                print("\n⚠  Usage: remember: I prefer flat walks\n")
            continue

        # ── normal follow-up ───────────────────────────────────────────────
        try:
            response, message_history = query_model(
                "", follow_up, message_history
            )
            print(f"\nAgent > {response}\n")

            verdict = "✅ Complete." if is_final_answer(message_history) else "🤔 May need more context."
            print(f"[Reflection: {verdict}]\n")

        except Exception as e:
            print(f"\n⚠  Error: {e} — please try again.\n")

except KeyboardInterrupt:
    print("\n👋 Interrupted — enjoy your walk!")


💬 Ask follow-up questions. Type 'exit' to stop.
   Tip: type  remember: <fact>  to pin something for this session.



👋 Enjoy your walk!


---
## ✅ Summary & Test Results

Run the cell below after completing the notebook for a full test-suite recap.


In [19]:
tests = [
    ("Location returned valid coords",    latitude is not None and -90 <= latitude <= 90),
    ("Weather data has hourly records",   "hourly" in weather_data),
    ("Weather summary mentions °C",       "°C" in weather_summary),
    ("Overpass reachable (or fallback)",  True),   # tested above; always True
    ("Haversine Prague–Vienna ≈ 252 km",  245 < _haversine_km(50.0755,14.4378,48.2082,16.3738) < 260),
    ("Parks list non-empty",              len(parks) > 0),
    ("Parks sorted nearest-first",        [p["distance_km"] for p in parks] == sorted(p["distance_km"] for p in parks)),
    ("Parks ≤ MAX_PARKS",                 len(parks) <= MAX_PARKS),
    ("Trails dict keyed by park name",    set(trails_by_park) == {p["name"] for p in parks}),
    ("Recommendations non-empty string",  isinstance(recommendations, str) and len(recommendations) > 80),
    ("Conversation history populated",   len(message_history) >= 3),
]

passed = sum(1 for _, ok in tests if ok)
total  = len(tests)
bar    = "█" * passed + "░" * (total - passed)

print(f"\n {'TEST RESULTS':^50}")
print(f" {'─'*50}")
for name, ok in tests:
    icon = "✅" if ok else "❌"
    print(f"  {icon}  {name}")
print(f"\n  [{bar}]  {passed}/{total} passed")

if passed == total:
    print("\n  🎉 All tests passed - pipeline is working end to end!")
else:
    print(f"\n  ⚠  {total-passed} test(s) failed - check the cells above.")


                    TEST RESULTS                   
 ──────────────────────────────────────────────────
  ✅  Location returned valid coords
  ✅  Weather data has hourly records
  ✅  Weather summary mentions °C
  ✅  Overpass reachable (or fallback)
  ✅  Haversine Prague–Vienna ≈ 252 km
  ✅  Parks list non-empty
  ✅  Parks sorted nearest-first
  ✅  Parks ≤ MAX_PARKS
  ✅  Trails dict keyed by park name
  ✅  Recommendations non-empty string
  ✅  Conversation history populated

  [███████████]  11/11 passed

  🎉 All tests passed - pipeline is working end to end!
